# Module `visualization.py`

Ce notebook montre comment visualiser les graphes CESIPATH avec `matplotlib`. Il couvre les trois vues (base, residuel, dynamique) et la session interactive.

Attention aux boutons dans Jupyter : selon le backend, le bouton `->` peut etre capricieux. Pour une utilisation intensive, preferer `python3 main_visualization.py`.

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig
from cesipath.visualization import GraphVisualizer

## 1. Semantique des couleurs

Constantes exposees par `GraphVisualizer` :

| Constante | Couleur | Signification |
|---|---|---|
| `BASE_EDGE_COLOR` | `#7a7a7a` gris | Route de base (vue statique neutre) |
| `ACTIVE_EDGE_COLOR` | `#2a9d8f` vert | Route libre ou active |
| `SURCHARGED_EDGE_COLOR` | `#7b2cbf` violet | Route surchargee |
| `FORBIDDEN_EDGE_COLOR` | `#c1121f` rouge pointille | Route interdite statiquement |
| `TEMP_DISABLED_EDGE_COLOR` | `#f4a261` orange pointille | Route temporairement OFF (dynamique) |
| `DEPOT_COLOR` | `#264653` bleu fonce | Sommet depot |
| `CLIENT_COLOR` | `#8ecae6` bleu clair | Sommet client |

Les etiquettes sur les aretes affichent le cout courant (statique ou dynamique selon la vue). Pour les interdictions, le texte est `interdit` ; pour les OFF dynamiques, le texte est `OFF`.

## 2. Generation d'une instance a visualiser

On prend une taille modeste (8 sommets) pour que les labels de couts restent lisibles.

In [ ]:
config = GraphGenerationConfig(
    node_count=8,
    seed=42,
    auto_density_profile=True,
    forbidden_rate=0.1,
    surcharge_rate=0.25,
    dynamic_sigma=0.18,
    dynamic_mean_reversion_strength=0.35,
    dynamic_max_multiplier=1.8,
    dynamic_forbid_probability=0.05,
    dynamic_restore_probability=0.2,
    dynamic_max_disabled_ratio=0.2,
)

generator = GraphGenerator(config)
instance = generator.generate()
visualizer = GraphVisualizer(instance, generator)
instance.summary()

## 3. `show_base_graph` - graphe physique brut

Affiche uniquement les vraies routes physiques avec leur cout de base. Aucune contrainte statique n'est appliquee a ce stade. C'est la vue de reference avant toute analyse.

In [ ]:
visualizer.show_base_graph()

## 4. `show_residual_graph` - apres contraintes statiques

Affiche le graphe apres application des interdictions et surcouts statiques. Cette vue montre exactement ce que Dijkstra voit pour construire la matrice metrique.

In [ ]:
visualizer.show_residual_graph()

## 5. `show_dynamic_graph` - session interactive

Retourne un `GraphVisualizationSession` qui contient :

- le simulateur dynamique ;
- le snapshot courant ;
- la figure matplotlib et son bouton `->`.

Le titre de la figure affiche en temps reel : numero de tour, aretes actives, densite active, ratio OFF.

In [ ]:
session = visualizer.show_dynamic_graph()
session.fig

## 6. Fallback : avancer manuellement

Si le bouton `->` ne reagit pas dans votre frontend Jupyter (par exemple avec `%matplotlib inline`), la methode `advance_session` fait la meme chose en code.

In [ ]:
visualizer.advance_session(session)
session.fig

## 7. Visualisation d'une solution de tournee

Le sous-module `cesipath.algorithms.visualization` expose deux fonctions utiles :

- `plot_solution(instance, solution)` : trace les tournees sur le graphe, une couleur par vehicule, sur fond du graphe residuel.
- `save_solution_plot(...)` : enregistre la figure dans `src/cesipath/algorithms/image/` avec un numero auto-incremente (partage avec `save_benchmark_figures`). Les fichiers generes sont ignores par `.gitignore`.

On lance un mini solveur pour la demo.

In [ ]:
from cesipath.algorithms import grasp, plot_solution
from cesipath.solver_input import build_static_solver_input

solver_input = build_static_solver_input(instance)
solution = grasp(solver_input, max_iterations=5, rcl_alpha=0.2, seed=42)
plot_solution(instance, solution)

## 8. Pourquoi pas d'accents dans les labels ?

Le projet suit une convention historique : commentaires et chaines en francais **sans accents**. Cela evite les problemes d'encodage sur les vieilles consoles ou les environnements mal configures. Les labels matplotlib sont soumis a la meme regle.